![Portfolio illustration for the automotive RAG assistant](Gemini_Generated_Image_bi58a7bi58a7bi58.png)

# Automotive Warning Assistant with RAG

This notebook demonstrates a Retrieval-Augmented Generation (RAG) workflow for an in-vehicle assistant. The goal is to answer driver questions about dashboard warning messages by grounding the response in a car manual instead of relying only on the model's general knowledge.

The source document is `mg-zs-warning-messages.html`, which contains warning lights, meanings, and recommended actions for the MG ZS. The notebook loads that document, chunks it, stores embeddings in Chroma, retrieves the most relevant passages, and asks a locally hosted LLM to produce a concise answer.

**Why this makes a good portfolio project:** it shows practical LLM engineering skills such as document ingestion, chunking strategy, vector search, prompt design, and building a lightweight end-to-end question-answering pipeline.

**Note:** this version uses a local LM Studio server through its OpenAI-compatible API. Make sure LM Studio is running and that both a chat model and an embedding model are loaded before executing the notebook.

## Workflow Overview

We will move through five simple stages:

1. Load the HTML manual into LangChain documents.
2. Split the manual into chunks that can be embedded and retrieved.
3. Index those chunks in a Chroma vector store.
4. Retrieve the most relevant chunks for a user question.
5. Generate a grounded answer with a local chat model.

This keeps the notebook easy to explain in interviews while still covering the core mechanics of a production-style RAG pipeline.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [2]:
# Load the MG ZS warning-message manual from the local HTML file.
loader = UnstructuredHTMLLoader(file_path="mg-zs-warning-messages.html")
car_docs = loader.load()

## 1. Load and Prepare the Source Document

The manual arrives as one long HTML document. Before retrieval works well, we need to turn it into smaller overlapping text chunks so the retriever can return focused context instead of the entire manual.

In [3]:
# LM Studio must be running locally with these models loaded.
# The chat model writes the final answer, while the embedding model powers retrieval.
LM_STUDIO_BASE_URL = "http://127.0.0.1:1234/v1"
LM_STUDIO_CHAT_MODEL = "qwen/qwen3.5-35b-a3b"
LM_STUDIO_EMBEDDING_MODEL = "text-embedding-mxbai-embed-large-v1"

# LM Studio exposes an OpenAI-compatible API and accepts any placeholder key.
LM_STUDIO_API_KEY = "lm-studio"

llm = ChatOpenAI(
    model=LM_STUDIO_CHAT_MODEL,
    temperature=0,
    openai_api_base=LM_STUDIO_BASE_URL,
    openai_api_key=LM_STUDIO_API_KEY,
)
embeddings = OpenAIEmbeddings(
    model=LM_STUDIO_EMBEDDING_MODEL,
    openai_api_base=LM_STUDIO_BASE_URL,
    openai_api_key=LM_STUDIO_API_KEY,
    check_embedding_ctx_length=False
)

In [4]:
car_docs

[Document(metadata={'source': 'mg-zs-warning-messages.html'}, page_content='Warning Message Procedure Cruise Control Fault Indicates that the cruise control system has detected a fault. Please consult an MG Authorised Repairer as soon as possible. Active Speed Limiter Fault Indicates that the active speed limit system has detected a fault. Contact an MG Authorised Repairer as soon as possible. Engine Coolant Temperature High High engine coolant temperature could result in severe damage. As soon as conditions permit, safely stop the vehicle and switch off the engine and contact an MG Authorised Repairer immediately. Engine Coolant Temperature Sensor Fault Indicates that the engine coolant temperature sensor has failed. As soon as conditions permit, safely stop the vehicle and switch off the engine and contact an MG Authorised Repairer immediately.\n\nWarning Message Procedure Low Oil Pressure Indicates that the oil pressure is too low, which may result in severe engine damage. As soon a

In [5]:
# Split the manual into overlapping chunks.
# Overlap helps preserve meaning when a warning description spans chunk boundaries.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", ""])


docs = splitter.split_documents(car_docs)
print(docs[0])

page_content='Warning Message Procedure Cruise Control Fault Indicates that the cruise control system has detected a fault. Please consult an MG Authorised Repairer as soon as possible. Active Speed Limiter Fault Indicates that the active speed limit system has detected a fault. Contact an MG Authorised Repairer as soon as possible' metadata={'source': 'mg-zs-warning-messages.html'}


## 2. Build Retrieval

After chunking, we convert the chunks into embeddings and store them in Chroma. At query time, similarity search pulls back the chunks that are most likely to answer the driver's question.

In [6]:
# Store chunk embeddings in Chroma so we can run semantic search over the manual.
vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings
)


In [7]:
# Return the top 2 most relevant chunks for each question.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [8]:
# Keep the answer short and grounded in the retrieved context.
prompt = ChatPromptTemplate.from_template("You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")

In [9]:
# Chain retrieval and generation together into one callable pipeline.
rag_chain = ({"context": retriever, "question": RunnablePassthrough()}
             | prompt
             | llm)

## 3. Run a Sample Query

The example below simulates a driver asking what a warning message means and what action they should take. Because the answer is grounded in the manual, the model stays focused on the source text instead of improvising.

In [10]:
# Example user question for the assistant.
query = "The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?"

In [11]:
# Run the full RAG pipeline and print the grounded answer.
answer = rag_chain.invoke(query).content
print(answer)



The warning indicates that your gasoline particular filter is full. You should consult an MG Authorised Repairer as soon as possible to address this issue. This ensures the vehicle receives the necessary professional service.


## Suggested Next Steps

- Add more vehicle documentation such as maintenance schedules, dashboard symbols, and troubleshooting pages.
- Persist the Chroma database to disk so the index does not need to be rebuilt each session.
- Add source citations to each answer so users can see which manual excerpt was used.
- Wrap the chain in a simple web app or voice interface to make the demo feel closer to an in-car assistant.

These extensions would turn the notebook from a proof of concept into a stronger product-style demo.